## Imports

In [3]:
import sys
sys.path.append('Desktop/Kaitlyn_Catalyst/cameratrapai/speciesnet')
from speciesnet.classifier import SpeciesNetClassifier
from speciesnet.detector import SpeciesNetDetector
from speciesnet.constants import Detection
from speciesnet.ensemble_prediction_combiner import combine_predictions_for_single_item
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageDraw
%matplotlib inline
import matplotlib.pyplot as plt
import torch.nn as nn

In [ ]:
# # Get the number of new total classes
# model = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
# # original_num_classes = len(model.labels)  # Original classes from speciesnet_label.json
# # new_labels = ["Mongoose"]

## Download models

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.model_download("google/speciesnet/pyTorch/v4.0.1a")

# print("Path to model files:", path)

## Initializing

In [5]:
# === Paths ===
image_path = "/Users/sarahabdelazim/Desktop/Kaitlyn_Catalyst/ct_classifier/datasets/all_species_images/496.JPG"
img = Image.open(image_path).convert("RGB")

# === Device Setup ===
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## Detection

In [ ]:
detector_model_name = "/Users/sarahabdelazim/.cache/kagglehub/models/google/speciesnet/pyTorch/v4.0.1a/1"

# === Create an Instance of Detector ===
detector = SpeciesNetDetector(detector_model_name)
preprocessed_image = detector.preprocess(img)

# === Run Detection ===
detections_result = detector.predict(filepath=image_path, img=preprocessed_image)

# === Display Detection Results ===
print("Detections:")
for detection in detections_result.get("detections", []):
    print(
        f"Category: {detection['category']}, "
        f"Label: {detection['label']}, "
        f"Confidence: {detection['conf']:.2f}, "
        f"BBox: {detection['bbox']}"
    )

# === Handle Failures ===
if "failures" in detections_result:
    print("Detection failed for the following reasons:", detections_result["failures"])

draw = ImageDraw.Draw(img)

# Iterate over detections
for detection in detections_result.get("detections", []):
    bbox = detection['bbox']  # [x_min, y_min, width, height]
    
    # Convert normalized bbox (relative coordinates) to absolute pixel values
    x_min = int(bbox[0] * img.width)
    y_min = int(bbox[1] * img.height)
    # Correctly calculate x_max and y_max
    x_max = int((bbox[0] + bbox[2]) * img.width)
    y_max = int((bbox[1] + bbox[3]) * img.height)

    # Debugging: Print bounding box values
    print(f"Normalized bbox: {bbox}")
    print(f"Absolute bbox: x_min={x_min}, y_min={y_min}, x_max={x_max}, y_max={y_max}")

    # Skip invalid bounding boxes
    if x_max < x_min or y_max < y_min:
        print(f"Skipping invalid bbox: x_min={x_min}, y_min={y_min}, x_max={x_max}, y_max={y_max}")
        continue

    # Draw rectangle (bounding box)
    draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)

    # Optional: Add label and confidence score
    label = f"{detection['label']} ({detection['conf']:.2f})"
    draw.text((x_min, y_min), label, fill="red")

# === Display Image with Bounding Boxes in Jupyter ===
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')  # Turn off axes for better visualization
plt.show()

## Classification

In [1]:
import torch
import torch.nn as nn

class AugmentedSpeciesNet(nn.Module):
    def __init__(self, base_model, original_outputs, extra_outputs=1):
        super().__init__()
        self.base_model = base_model
        self.extra_head = nn.Linear(original_outputs, original_outputs + extra_outputs)

    def forward(self, x):
        base_logits = self.base_model(x)  # output shape: [batch, original_outputs]
        augmented_logits = self.extra_head(base_logits)
        return augmented_logits

In [7]:
# === Configuration ===
classifier_model_name = "/Users/sarahabdelazim/.cache/kagglehub/models/google/speciesnet/pyTorch/v4.0.1a/1"
target_species_txt = "/Users/sarahabdelazim/Desktop/Kaitlyn_Catalyst/ct_classifier/target_species.txt"


# === Preprocessing ===
transform = transforms.Compose([
    transforms.Resize((480, 480)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load image and apply transform
img = Image.open(image_path).convert("RGB")
img_tensor = transform(img)  # ✅ apply the transform
x = img_tensor.unsqueeze(0).to(device)  # ✅ shape = [1, 3, 480, 480]
print(f"Final input shape: {x.shape}")

# === Load model with target labels ===
model = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
original_outputs = len(model.target_labels)
model.model = AugmentedSpeciesNet(model.model, original_outputs=original_outputs)
model.model = model.model.to(device)

# Load target labels and their indices
target_labels = model.target_labels  # List of target labels
target_indices = model.target_idx    # Indices corresponding to the target labels
# print(f"Target Labels: {target_labels}")

# === Prediction ===
with torch.no_grad():
    logits = model.model(x)
    print("✅ Output shape from base model:", logits.shape)
    probs = F.softmax(logits, dim=1)

    # Filter logits and probabilities for target labels
    target_logits = logits[0, target_indices]  # Logits for target labels
    target_probs = F.softmax(target_logits, dim=0)  # Normalize probabilities for target labels

    # Get top-k predictions for target labels
    topk = 5
    target_probs, target_indices = torch.topk(target_probs, topk)
    top_classes = [target_labels[idx.item()] for idx in target_indices]
    top_scores = [target_probs[i].item() for i in range(topk)]

    classifications = {
        "classes": top_classes,  # List of top predicted target classes
        "scores": top_scores     # List of confidence scores for each target class
    }

# === Output Predictions ===
print("🔝 Top-5 Predictions (Target Labels):")
for i in range(topk):
    print(f"{i+1}. {top_classes[i]} ({top_scores[i]*100:.2f}%)")

Final input shape: torch.Size([1, 3, 480, 480])


RuntimeError: Given groups=1, weight of size [24, 3, 3, 3], expected input[1, 480, 4, 481] to have 3 channels, but got 480 channels instead

In [23]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

# === Custom wrapper to add 1 new class
class AugmentedSpeciesNet(nn.Module):
    def __init__(self, base_model, original_outputs, extra_outputs=1):
        super().__init__()
        self.base_model = base_model
        self.extra_head = nn.Linear(original_outputs, original_outputs + extra_outputs)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1).contiguous()  # [B, C, H, W] → [B, H, W, C]
        with torch.no_grad():
            base_logits = self.base_model(x)  # [B, original_outputs]
        return self.extra_head(base_logits)   # [B, original_outputs + 1]

# === Load base classifier
classifier = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
original_outputs = len(classifier.labels)

# === Replace model with augmented version
classifier.model = AugmentedSpeciesNet(classifier.model, original_outputs, extra_outputs=1)
classifier.model = classifier.model.to(device)

# === Load and preprocess image
img = Image.open(image_path).convert("RGB")
preprocessed = classifier.preprocess(img)
arr = preprocessed.arr / 255.0  # normalize
x = torch.tensor(arr).unsqueeze(0).float().to(device)  # [1, H, W, C]
x = x.permute(0, 3, 1, 2)  # → [1, C, H, W]

print("🚀 Input shape:", x.shape)

# === Run inference
with torch.no_grad():
    logits = classifier.model(x)  # [1, original_outputs + 1]

    # === Filter to target species + Mongoose
    mongoose_index = original_outputs
    target_indices = classifier.target_idx + [mongoose_index]  # all target class indices + Mongoose
    target_logits = logits[0, target_indices]  # select relevant logits
    target_probs = F.softmax(target_logits, dim=0)

    # Get top-5 among filtered classes
    topk = min(5, len(target_indices))
    top_probs, top_indices = torch.topk(target_probs, topk)

    # Combine original target labels + new label
    filtered_labels = classifier.target_labels + ["Mongoose"]
    top_classes = [filtered_labels[idx.item()] for idx in top_indices]
    top_scores = [top_probs[i].item() for i in range(topk)]

# === Output Top-5
print("🔝 Top-5 Predictions:")
for i, (label, score) in enumerate(zip(top_classes, top_scores)):
    print(f"{i+1}. {label} ({score*100:.2f}%)")

🚀 Input shape: torch.Size([1, 3, 480, 480])
🔝 Top-5 Predictions:
1. 7631afd3-ab8e-4e88-9243-2c21386595eb;mammalia;cetartiodactyla;hippopotamidae;hippopotamus;amphibius;hippopotamus (34.17%)
2. 9732cefb-6a08-49f6-b61e-b9a9054368c4;mammalia;cetartiodactyla;bovidae;syncerus;caffer;african buffalo (29.86%)
3. 7bfdcb3a-386b-4e47-9780-3fea47fa4e6e;mammalia;cetartiodactyla;bovidae;tragelaphus;angasii;nyala (17.55%)
4. 6130285b-ff22-4602-b7e8-ef92e3a527da;mammalia;cetartiodactyla;suidae;potamochoerus;larvatus;bushpig (7.64%)
5. 341cda2b-34df-4391-a61e-ba063bbe2f9a;mammalia;carnivora;herpestidae;herpestes;ichneumon;egyptian mongoose (4.12%)


## Ensemble

In [ ]:
# country = "MOZ"  # Optional country information
# admin1_region = "Sofala"  # Optional region information

# # Maps and configuration
# taxonomy_map = {}  # Define your taxonomy mapping
# geofence_map = {}  # Define your geofence mapping
# enable_geofence = True

# # Define geofencing and roll-up functions
# def geofence_fn(labels, scores, country, admin1_region, taxonomy_map, geofence_map, enable_geofence):
#     # Example geofencing logic
#     return labels[0], scores[0], "geofence"

# def roll_up_fn(labels, scores, country, admin1_region, target_taxonomy_levels, non_blank_threshold, taxonomy_map, geofence_map, enable_geofence):
#     # Example roll-up logic
#     return labels[0], scores[0], "rollup"

# # === Combine Predictions ===
# ensemble_result = combine_predictions_for_single_item(
#     classifications=classifications,
#     detections = detections_result.get("detections", []),
#     country=country,
#     admin1_region=admin1_region,
#     taxonomy_map=taxonomy_map,
#     geofence_map=geofence_map,
#     enable_geofence=enable_geofence,
#     geofence_fn=geofence_fn,
#     roll_up_fn=roll_up_fn
# )

# # === Output the Result ===
# label, score, source = ensemble_result
# print(f"Ensembled Prediction: {label} (Confidence: {score}, Source: {source})")

## Get Layer Names

In [ ]:
# # Retrieve and print the layer names
# layer_names = [name for name, _ in classifier.model.named_modules()]
# print("Layer Names:")
# for name in layer_names:
#     print(name)